# Reference analysis notebook

This notebook is provided as reference analysis code for the accepted paper figures. It is not intended to be a standalone reproduction package. Model weights, LoRA adapters, datasets, and intermediate hidden-state files are not included. Local paths under `data/`, `adapters/`, and `outputs/` should be adjusted to the user's environment.

The output key `mutual_information_to_final` is retained for compatibility with the original analysis outputs. The plotted label uses `Gaussian pseudo-MI to Final Layer` because this quantity is a Gaussian/log-det diagnostic, not a calibrated mutual-information estimator.


In [ ]:
# Figures: layer-wise effective rank, adjacent-layer alignment, log-det covariance, and Gaussian pseudo-MI to the final layer.
# Required inputs: a base model, adapter paths, and a prompt used to extract hidden states.
# The notebook computes hidden-state metrics for each adapter and saves one figure per metric.

from pathlib import Path
import gc
import numpy as np
import torch
from torch.linalg import svd, svdvals
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
PROMPT = "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n### Instruction: Please answer the following question with true or false, question: do the courts have the power to overrule the laws written by the legislative branch of government?\n\nAnswer format: true/false\n### Response:"
MODEL_SPECS = [
    {"label": "Baseline", "adapter_path": Path("adapters/baseline")},
    {"label": "Ours", "adapter_path": Path("adapters/ours")},
]
OUTPUT_DIR = Path("outputs/layerwise_representation_metrics")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16
MERGE_LORA = False
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def finite_rows(x):
    return x[torch.isfinite(x).all(dim=1)]

def effective_rank(x):
    x = finite_rows(x.reshape(-1, x.size(-1)))
    if x.shape[0] < 2:
        return float("nan")
    x = x - x.mean(dim=0, keepdim=True)
    cov = (x.T @ x) / (x.shape[0] - 1)
    values = svd(cov, full_matrices=False).S
    values = values[values > 1e-8]
    probs = values / values.sum()
    return torch.exp(-(probs * torch.log(probs)).sum()).item()

def alignment_cosine(prev, curr):
    prev = prev.reshape(-1, prev.size(-1))
    curr = curr.reshape(-1, curr.size(-1))
    delta = curr - prev
    num = (prev * delta).sum(dim=1)
    denom = prev.norm(dim=1) * delta.norm(dim=1) + 1e-8
    return (num / denom).mean().item()

def logdet_covariance(x):
    x = finite_rows(x.reshape(-1, x.size(-1)))
    if x.shape[0] < 2:
        return float("nan")
    x = x - x.mean(dim=0, keepdim=True)
    cov = (x.T @ x) / (x.shape[0] - 1)
    values = svdvals(cov)
    values = values[values > 1e-8]
    return values.log().sum().item()

def gaussian_mutual_information(x, y):
    x = finite_rows(x.reshape(-1, x.size(-1)))
    y = finite_rows(y.reshape(-1, y.size(-1)))
    if x.shape[0] != y.shape[0] or x.shape[0] < 2:
        return float("nan")
    def logdet(z):
        z = z - z.mean(dim=0, keepdim=True)
        cov = (z.T @ z) / (z.shape[0] - 1)
        cov = cov + 1e-5 * torch.eye(cov.size(0), device=cov.device)
        values = svdvals(cov)
        values = values[values > 1e-8]
        return values.log().sum()
    return (0.5 * (logdet(x) + logdet(y) - logdet(torch.cat([x, y], dim=1)))).item()

def hidden_states_for_adapter(adapter_path):
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=DTYPE, trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if adapter_path is not None:
        if not Path(adapter_path).exists():
            raise FileNotFoundError(f"Adapter path not found: {adapter_path}")
        model = PeftModel.from_pretrained(base_model, str(adapter_path))
        if MERGE_LORA:
            model = model.merge_and_unload()
    else:
        model = base_model
    model = model.to(DEVICE).eval()
    inputs = tokenizer(PROMPT, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    if outputs.hidden_states is None:
        raise RuntimeError("Model did not return hidden states")
    hidden_states = [h[0].detach().float().cpu() for h in outputs.hidden_states[1:]]
    del model, base_model, outputs
    torch.cuda.empty_cache()
    gc.collect()
    return hidden_states

In [ ]:
def compute_metrics(hidden_states):
    ranks = [effective_rank(h) for h in hidden_states]
    alignments = [alignment_cosine(hidden_states[i], hidden_states[i + 1]) for i in range(len(hidden_states) - 1)]
    logdets = [logdet_covariance(h) for h in hidden_states[:-1]]
    mutual_infos = [gaussian_mutual_information(h, hidden_states[-1]) for h in hidden_states[:-1]]
    return {"effective_rank": ranks, "alignment_cosine": alignments, "logdet_covariance": logdets, "mutual_information_to_final": mutual_infos}

def plot_metric(results, metric, ylabel, output_name):
    plt.figure(figsize=(8, 5))
    for label, metrics in results.items():
        values = np.asarray(metrics[metric], dtype=np.float64)
        xs = np.arange(1, len(values) + 1)
        mask = np.isfinite(values)
        plt.plot(xs[mask], values[mask], label=label, linewidth=2.5, marker="o")
    plt.xlabel("Layer")
    plt.ylabel(ylabel)
    plt.title(ylabel)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / output_name, dpi=220, bbox_inches="tight")
    plt.show()

In [ ]:
results = {}
for spec in MODEL_SPECS:
    hidden_states = hidden_states_for_adapter(spec["adapter_path"])
    results[spec["label"]] = compute_metrics(hidden_states)

for label, metrics in results.items():
    np.savez(OUTPUT_DIR / f"{label.lower()}_metrics.npz", **{k: np.asarray(v, dtype=np.float64) for k, v in metrics.items()})

plot_metric(results, "effective_rank", "Effective Rank", "effective_rank.png")
plot_metric(results, "alignment_cosine", "Alignment Cosine", "alignment_cosine.png")
plot_metric(results, "logdet_covariance", "Log-Det Covariance", "logdet_covariance.png")
plot_metric(results, "mutual_information_to_final", "Gaussian pseudo-MI to Final Layer", "mutual_information_to_final.png")